# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [7]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [8]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dates = pd.to_datetime(spotify_raw['Release Date'])
dim_tiempo = pd.DataFrame({
    'Release Date': spotify_raw['Release Date'],
    'Year': dates.dt.year,
    'Month': dates.dt.month,
    'Day': dates.dt.day
}).drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1
dim_tiempo = dim_tiempo[['id_tiempo', 'Year', 'Month', 'Day', 'Release Date']]
display(dim_tiempo.head())


,id_tiempo,Year,Month,Day,Release Date
0,1,2024,4,26,4/26/2024
1,2,2024,5,4,5/4/2024
2,3,2024,3,19,3/19/2024
3,4,2023,1,12,1/12/2023
4,5,2024,5,31,5/31/2024


In [9]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_pista'] = dim_pista.index + 1
dim_pista = dim_pista[['id_pista', 'Track', 'Artist', 'Explicit Track']]
display(dim_pista.head())


,id_pista,Track,Artist,Explicit Track
0,1,MILLION DOLLAR BABY,Tommy Richman,0
1,2,Not Like Us,Kendrick Lamar,1
2,3,i like the way you kiss me,Artemas,0
3,4,Flowers,Miley Cyrus,0
4,5,Houdini,Eminem,1


In [10]:
# 3. Crear fact_streams
metric_cols = [
    'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count',
    'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes',
    'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach',
    'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins',
    'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count',
    'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams',
    'Shazam Counts', 'TIDAL Popularity'
]

spotify_clean = spotify_raw.copy()
for col in metric_cols:
    spotify_clean[col] = pd.to_numeric(spotify_clean[col].astype(str).str.replace(',', ''), errors='coerce')

df_temp = spotify_clean.merge(dim_tiempo, on='Release Date', how='left')
df_temp = df_temp.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left')

fact_streams = df_temp[['id_pista', 'id_tiempo'] + metric_cols].copy()
display(fact_streams.head())


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity
0,1,1,1,725.4,3.904709e+08,30716.0,196631588.0,92.0,8.427475e+07,1713126.0,...,40975.0,684.0,62.0,17598718.0,114.0,18004655.0,22931.0,4818457.0,2669262.0,NaN
1,2,2,2,545.9,3.237039e+08,28113.0,174597137.0,92.0,1.163470e+08,3486739.0,...,40778.0,3.0,67.0,10422430.0,111.0,7780028.0,28444.0,6623075.0,1118279.0,NaN
2,3,3,3,538.4,6.013093e+08,54331.0,211607669.0,92.0,1.225991e+08,2228730.0,...,74333.0,536.0,136.0,36321847.0,172.0,5022621.0,5639.0,7208651.0,5285340.0,NaN
3,4,4,4,444.9,2.031281e+09,269802.0,136569078.0,85.0,1.096101e+09,10629796.0,...,1474799.0,2182.0,264.0,24684248.0,210.0,190260277.0,203384.0,NaN,11822942.0,NaN
4,5,5,5,423.3,1.070349e+08,7223.0,151469874.0,88.0,7.737396e+07,3670188.0,...,12185.0,1.0,82.0,17660624.0,105.0,4493884.0,7006.0,207179.0,457017.0,NaN


In [11]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_pista, on='id_pista', how='inner').merge(dim_tiempo, on='id_tiempo', how='inner')
display(cubo_olap.head())


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Year,Month,Day,Release Date
0,1,1,1,725.4,3.904709e+08,30716.0,196631588.0,92.0,8.427475e+07,1713126.0,...,4818457.0,2669262.0,NaN,MILLION DOLLAR BABY,Tommy Richman,0,2024,4,26,4/26/2024
1,2,2,2,545.9,3.237039e+08,28113.0,174597137.0,92.0,1.163470e+08,3486739.0,...,6623075.0,1118279.0,NaN,Not Like Us,Kendrick Lamar,1,2024,5,4,5/4/2024
2,3,3,3,538.4,6.013093e+08,54331.0,211607669.0,92.0,1.225991e+08,2228730.0,...,7208651.0,5285340.0,NaN,i like the way you kiss me,Artemas,0,2024,3,19,3/19/2024
3,4,4,4,444.9,2.031281e+09,269802.0,136569078.0,85.0,1.096101e+09,10629796.0,...,NaN,11822942.0,NaN,Flowers,Miley Cyrus,0,2023,1,12,1/12/2023
4,5,5,5,423.3,1.070349e+08,7223.0,151469874.0,88.0,7.737396e+07,3670188.0,...,207179.0,457017.0,NaN,Houdini,Eminem,1,2024,5,31,5/31/2024


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [12]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum().reset_index()
display(respuesta_1)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
display(respuesta_2)

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down
respuesta_3 = cubo_olap.groupby(['Year', 'Month'])['Spotify Streams'].sum().reset_index()
display(respuesta_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['Year'] == 2024)]
display(respuesta_4)

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up
respuesta_5 = cubo_olap.groupby('Year')['Track Score'].mean().reset_index()
display(respuesta_5)


,Artist,Spotify Streams
0,"""XY""",0.000000e+00
1,$OHO BANI,5.406556e+07
2,$uicideboy$,1.697447e+09
3,&ME,3.460163e+07
4,(G)I-DLE,8.769385e+08
...,...,...
1994,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,2.409532e+06
1995,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),5.484080e+05
1996,ýýýýýýýýýýýýýýýýýý,4.943057e+06
1997,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,4.560340e+05


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Year,Month,Day,Release Date
0,1,1,1,725.4,390470936.0,30716.0,196631588.0,92.0,84274754.0,1713126.0,...,4818457.0,2669262.0,NaN,MILLION DOLLAR BABY,Tommy Richman,0,2024,4,26,4/26/2024
1,2,2,2,545.9,323703884.0,28113.0,174597137.0,92.0,116347040.0,3486739.0,...,6623075.0,1118279.0,NaN,Not Like Us,Kendrick Lamar,1,2024,5,4,5/4/2024
2,3,3,3,538.4,601309283.0,54331.0,211607669.0,92.0,122599116.0,2228730.0,...,7208651.0,5285340.0,NaN,i like the way you kiss me,Artemas,0,2024,3,19,3/19/2024
4,5,5,5,423.3,107034922.0,7223.0,151469874.0,88.0,77373957.0,3670188.0,...,207179.0,457017.0,NaN,Houdini,Eminem,1,2024,5,31,5/31/2024
6,7,7,7,407.2,900158751.0,73118.0,201585714.0,86.0,308723145.0,4120760.0,...,NaN,9990302.0,NaN,Beautiful Things,Benson Boone,0,2024,1,18,1/18/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4544,4481,361,4523,19.6,6184259.0,1641.0,8841671.0,67.0,1090122.0,43929.0,...,373165.0,965.0,NaN,ATM,Nemzzz,1,2024,5,16,5/16/2024
4552,4489,13,4519,19.6,3970523.0,1230.0,392098.0,67.0,408201.0,9173.0,...,671893.0,NaN,NaN,AirBnb,ian,1,2024,5,17,5/17/2024
4561,4498,190,4547,19.5,1633647.0,160.0,2592948.0,NaN,9700826.0,164575.0,...,NaN,5598.0,NaN,"Tu Juliet Jatt Di ( From ""Jatt & Juliet 3"")",Diljit Dosanjh,0,2024,5,24,5/24/2024
4593,4530,616,4587,19.4,3754914.0,101.0,1306604.0,NaN,34315313.0,431343.0,...,NaN,3169.0,NaN,"Jaragandi (From ""Game Changer"") (Telugu)",Thaman S,0,2024,3,27,3/27/2024


,Year,Month,Spotify Streams
0,1987,7,1.879386e+09
1,1991,9,2.021910e+09
2,1994,1,1.810650e+09
3,1998,4,1.783399e+08
4,1999,6,1.405354e+09
...,...,...,...
186,2024,2,1.023077e+10
187,2024,3,1.263673e+10
188,2024,4,2.152035e+10
189,2024,5,1.005348e+10


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Year,Month,Day,Release Date
1,2,2,2,545.9,3.237039e+08,28113.0,174597137.0,92.0,116347040.0,3486739.0,...,6623075.0,1118279.0,NaN,Not Like Us,Kendrick Lamar,1,2024,5,4,5/4/2024
4,5,5,5,423.3,1.070349e+08,7223.0,151469874.0,88.0,77373957.0,3670188.0,...,207179.0,457017.0,NaN,Houdini,Eminem,1,2024,5,31,5/31/2024
7,8,8,8,375.8,6.750792e+08,40094.0,211236940.0,92.0,228382568.0,1439495.0,...,NaN,6063523.0,NaN,Gata Only,FloyyMenor,1,2024,2,2,2/2/2024
8,9,9,9,355.7,1.653018e+09,1.0,15.0,NaN,NaN,NaN,...,NaN,NaN,NaN,Danza Kuduro - Cover,MUSIC LAB JPN,1,2024,6,9,6/9/2024
9,10,10,10,330.6,9.067657e+07,10400.0,184199419.0,86.0,32735244.0,988682.0,...,3679709.0,666302.0,NaN,BAND4BAND (feat. Lil Baby),Central Cee,1,2024,5,23,5/23/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4486,4425,123,4494,19.7,1.594834e+07,4948.0,2021225.0,NaN,2127850.0,70258.0,...,NaN,101755.0,NaN,7 Minute Drill,J. Cole,1,2024,4,5,4/5/2024
4491,4430,132,4455,19.7,9.126485e+06,2054.0,1013145.0,NaN,2292547.0,82132.0,...,588983.0,NaN,NaN,Never Last,Lil Tecca,1,2024,5,3,5/3/2024
4544,4481,361,4523,19.6,6.184259e+06,1641.0,8841671.0,67.0,1090122.0,43929.0,...,373165.0,965.0,NaN,ATM,Nemzzz,1,2024,5,16,5/16/2024
4552,4489,13,4519,19.6,3.970523e+06,1230.0,392098.0,67.0,408201.0,9173.0,...,671893.0,NaN,NaN,AirBnb,ian,1,2024,5,17,5/17/2024


,Year,Track Score
0,1987,21.500000
1,1991,19.600000
2,1994,31.500000
3,1998,20.800000
4,1999,22.200000
5,2000,30.400000
6,2001,20.100000
7,2002,26.060000
8,2003,23.766667
9,2004,24.240000


In [13]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
display(respuesta_6)

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum().reset_index()
display(respuesta_7)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice
respuesta_8 = cubo_olap[(cubo_olap['Month'] == 12) & (cubo_olap['Year'] == 2023)]
display(respuesta_8)

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up
respuesta_9 = cubo_olap.groupby('Month')['Spotify Streams'].sum().reset_index()
display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice
respuesta_10 = cubo_olap[(cubo_olap['Artist'] == 'Taylor Swift') & (cubo_olap['Explicit Track'] == 0) & (cubo_olap['Year'] == 2024)]
display(respuesta_10)


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Year,Month,Day,Release Date
63,61,57,64,184.8,1.811991e+09,173358.0,67603671.0,79.0,1.516863e+09,8870768.0,...,NaN,3393920.0,NaN,Me Porto Bonito,Bad Bunny,1,2022,5,6,5/6/2022
88,86,57,89,163.1,1.514240e+09,190692.0,77283361.0,77.0,1.366181e+09,9358441.0,...,NaN,5133847.0,NaN,Titï¿½ï¿½ Me Pregu,Bad Bunny,1,2022,5,6,5/6/2022
102,100,91,103,150.7,4.781509e+08,73306.0,51842133.0,78.0,1.133797e+08,1312763.0,...,4374049.0,2255465.0,NaN,MONACO,Bad Bunny,1,2023,10,13,10/13/2023
174,171,91,175,114.2,6.891815e+08,61138.0,90492257.0,83.0,1.833815e+08,738123.0,...,2473108.0,930098.0,NaN,PERRO NEGRO,Bad Bunny,1,2023,10,13,10/13/2023
187,184,57,188,110.7,1.421763e+09,152592.0,62313059.0,78.0,9.458571e+08,5587691.0,...,NaN,3087489.0,NaN,Ojitos Lindos,Bad Bunny,0,2022,5,6,5/6/2022
245,242,204,246,98.4,7.178490e+08,98440.0,37797217.0,72.0,2.176877e+08,1929598.0,...,4037512.0,2076698.0,NaN,WHERE SHE GOES,Bad Bunny,1,2023,5,18,5/18/2023
261,258,135,262,96.1,1.977947e+09,245685.0,56430031.0,72.0,1.861597e+09,10999677.0,...,NaN,6091496.0,NaN,Dakiti,Bad Bunny,1,2020,1,1,1/1/2020
271,268,57,272,94.1,1.108032e+09,121256.0,44739826.0,77.0,7.657802e+08,4700638.0,...,9615729.0,1644044.0,NaN,Moscow Mule,Bad Bunny,1,2022,5,6,5/6/2022
290,287,57,292,90.1,1.293774e+09,115332.0,51081689.0,77.0,5.019082e+08,2730759.0,...,NaN,1927360.0,NaN,Efecto,Bad Bunny,1,2022,5,6,5/6/2022
294,291,238,295,88.6,1.398634e+09,177082.0,44848091.0,71.0,1.084872e+09,7894095.0,...,NaN,2211960.0,NaN,Yonaguni,Bad Bunny,1,2021,6,4,6/4/2021


,Artist,Track,YouTube Views
0,"""XY""",Come and Say - Original Mix,415159148.0
1,$OHO BANI,"ZEIT, DASS SICH WAS DREHT",6744201.0
2,$uicideboy$,"...And to Those I Love, Thanks for Sticking Ar...",269460083.0
3,$uicideboy$,1000 Blunts,87680985.0
4,$uicideboy$,"Are You Going to See the Rose in the Vase, or ...",2640169.0
...,...,...,...
4478,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý,44586424.0
4479,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),ýýýýýýýýýýýýýýýýýý,1356336.0
4480,ýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýýýýý,127116431.0
4481,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý,247148544.0


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Year,Month,Day,Release Date
61,59,55,62,186.5,1384.0,2.0,801.0,NaN,222076.0,5430.0,...,NaN,NaN,NaN,Very Cute Melody by Marimba Tone (39813),mitsu sound,0,2023,12,31,12/31/2023
94,92,84,95,155.9,649404552.0,27544.0,47789063.0,83.0,433839212.0,2750086.0,...,NaN,1380069.0,NaN,La Diabla,Xavi,0,2023,12,2,12/2/2023
97,95,87,98,153.1,359803580.0,26065.0,79152427.0,86.0,369321334.0,1839225.0,...,NaN,1002835.0,NaN,BELLAKEO,Peso Pluma,1,2023,12,7,12/7/2023
131,129,112,132,130.7,8327.0,3.0,8.0,NaN,NaN,NaN,...,NaN,522.0,NaN,Looking Like a Wow End of Year 2023,WasteLand,0,2023,12,10,12/10/2023
177,174,152,178,112.6,606124553.0,40628.0,139802900.0,86.0,166363445.0,1019744.0,...,NaN,1704574.0,NaN,LUNA,Feid,0,2023,12,1,12/1/2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4189,4133,979,4178,20.7,100634965.0,8790.0,29458776.0,NaN,19367095.0,200936.0,...,NaN,291674.0,NaN,Tacata (Remix),Tiagz,1,2023,12,22,12/22/2023
4194,4138,87,4179,20.7,9129729.0,1422.0,3777844.0,NaN,78290967.0,103564.0,...,NaN,71640.0,NaN,YA NO TE EXTRAï¿½,NATTI NATASHA,0,2023,12,7,12/7/2023
4385,4325,152,4374,20.1,45238166.0,6057.0,7158117.0,NaN,8478416.0,174490.0,...,NaN,321401.0,NaN,MY HOUSE,Beyoncï¿,1,2023,12,1,12/1/2023
4462,4401,1533,4425,19.8,3225791.0,411.0,11826.0,NaN,72510719.0,227398.0,...,NaN,25420.0,NaN,ýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý,0,2023,12,29,12/29/2023


,Month,Spotify Streams
0,1,3.652168e+11
1,2,1.333565e+11
2,3,1.638041e+11
3,4,1.462846e+11
4,5,1.779293e+11
5,6,1.484293e+11
6,7,1.503832e+11
7,8,1.426826e+11
8,9,1.363626e+11
9,10,1.672667e+11


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Year,Month,Day,Release Date
16,17,17,17,297.6,395433400.0,12784.0,177932568.0,89.0,104673836.0,2484083.0,...,377734.0,1210029.0,NaN,Fortnight (feat. Post Malone),Taylor Swift,0,2024,4,18,4/18/2024
866,861,17,862,52.3,166168092.0,5004.0,25607992.0,78.0,16423009.0,244859.0,...,NaN,116673.0,NaN,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0,2024,4,18,4/18/2024
1013,1007,17,1008,47.8,175613480.0,4454.0,20883678.0,84.0,15237337.0,222750.0,...,NaN,67886.0,NaN,"So Long, London",Taylor Swift,0,2024,4,18,4/18/2024
1167,1159,17,1161,44.0,161916445.0,3784.0,17036800.0,83.0,13509432.0,180327.0,...,NaN,47529.0,NaN,My Boy Only Breaks His Favorite Toys,Taylor Swift,0,2024,4,18,4/18/2024
1270,1260,17,1264,42.1,165611453.0,4418.0,21928475.0,78.0,9873283.0,96083.0,...,NaN,79964.0,NaN,Guilty as Sin?,Taylor Swift,0,2024,4,18,4/18/2024
2484,2454,183,2478,28.6,97601615.0,3397.0,13075013.0,87.0,5528058.0,78517.0,...,NaN,NaN,NaN,So High School,Taylor Swift,0,2024,4,19,4/19/2024
2517,2486,17,2513,28.3,106201248.0,2992.0,15108991.0,NaN,5521347.0,70034.0,...,NaN,24502.0,NaN,The Alchemy,Taylor Swift,0,2024,4,18,4/18/2024
2528,2497,17,2525,28.2,250430303.0,648.0,818594.0,NaN,NaN,NaN,...,NaN,NaN,NaN,I Can Do It With a Broken Heart,Taylor Swift,0,2024,4,18,4/18/2024
2776,2744,183,2764,26.7,88244010.0,2659.0,8584946.0,NaN,7381186.0,96742.0,...,NaN,32559.0,NaN,imgonnagetyouback,Taylor Swift,0,2024,4,19,4/19/2024
2816,2784,17,2811,26.5,105424914.0,2473.0,7785632.0,NaN,7227379.0,113318.0,...,NaN,21372.0,NaN,Fresh Out The Slammer,Taylor Swift,0,2024,4,18,4/18/2024
